In [37]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

In [ ]:
import os

# ----------------------------
# Imports
# ----------------------------

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [41]:
# ----------------------------
# API Key
# ----------------------------

#os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [43]:
# ----------------------------
# Embedding Model
# ----------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# ----------------------------
# Documents (Knowledge Base)
# ----------------------------

documents = [

Document(page_content="""
Machine learning is a field of artificial intelligence that focuses on building systems
that learn from data. It is widely used in recommendation systems, fraud detection,
and natural language processing.
"""),

Document(page_content="""
Deep learning is a subset of machine learning that uses neural networks with many layers.
It is particularly effective for image recognition, speech processing, and generative AI.
"""),

Document(page_content="""
Natural language processing (NLP) enables computers to understand human language.
It is used in chatbots, translation systems, and sentiment analysis.
"""),

Document(page_content="""
Reinforcement learning is a type of machine learning where agents learn by interacting
with an environment and receiving rewards or penalties.
""")
]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:



# ----------------------------
# Vector Database (FAISS)
# ----------------------------

vector_db = FAISS.from_documents(documents, embedding_model)

retriever = vector_db.as_retriever(search_kwargs={"k": 2})


# ----------------------------
# Tool (Example)
# ----------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))


tools = [calculator]


In [45]:


# ----------------------------
# LLM
# ----------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)


# ----------------------------
# Prompt Template
# ----------------------------

prompt = ChatPromptTemplate.from_template(
"""
You are an intelligent assistant.

Use the context below to answer the question.
If needed, you can use tools.

Chat History:
{chat_history}

Context:
{context}

Question:
{question}
"""
)


In [46]:


# ----------------------------
# Helper: format docs
# ----------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# ----------------------------
# RAG Chain (LCEL)
# ----------------------------

rag_chain = (
    {
        "context": RunnableLambda(lambda x: retriever.invoke(x["question"]))
                  | RunnableLambda(format_docs),
        "question": RunnableLambda(lambda x: x["question"]),
        "chat_history": RunnableLambda(lambda x: x["chat_history"]),
    }
    | prompt
    | llm_with_tools
)


In [47]:


# ----------------------------
# Memory
# ----------------------------

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)




Manual tool calling

In [48]:
while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    # Handle math explicitly
    if any(op in query for op in ["+", "-", "*", "/"]):
        try:
            expression = query.replace("=", "").replace("?", "").strip()
            result = eval(expression)
            print("\nAgent:", result)
            continue
        except:
            pass

    # Normal RAG flow
    response = agent.invoke(
        {"question": query},
        config={"configurable": {"session_id": "user1"}}
    )

    # Handle blank response safely
    if hasattr(response, "content") and response.content:
        print("\nAgent:", response.content)
    else:
        print("\n⚠️ No answer generated (likely tool call issue)")


You: What is machine learning?

Agent: [{'type': 'text', 'text': 'Machine learning is a field of artificial intelligence that focuses on building systems that learn from data. It is widely used in recommendation systems, fraud detection, and natural language processing.', 'extras': {'signature': 'CpsCAb4+9vtSFTgyltlJ9OPp09sds6SXxJjCo1JlH57Bze7iQgmY73mEcFm4HnJe3LtWOv4MS9RF4tp5lnsr+MZwQaBabYnLjzSNiIeHZPdnAI5SGkkgDbGmZvllW6BewVW1vw1pUPsCE6Ul8nn4qKsuEJuydd7uwmt1EbDBPx9QuFRk0Z6/LXeNci07KyFiWvB2svEl4cwWSJbLI8belurKFZ0vPf8rpUQmWFLe//BHWT+I3A2CZ3qbVxNjDmBvbyJ1OoCs7/0XfHQFEjO0I2205g2eGPAYPpyz28GmvrZhqvA4ededuRtAjFhcD26PgEooV84+u/lQlenPeBWrmQjldrhptG25s09A5wjZcH6ayamxXK92wXJ7o4b4wg=='}}]

You: what is deep learning

Agent: [{'type': 'text', 'text': 'Deep learning is a subset of machine learning that uses neural networks with many layers. It is particularly effective for image recognition, speech processing, and generative AI.', 'extras': {'signature': 'CoUDAb4+9vsGeIrOlFA92tvu7Sqe/6uWNECUmzKODA

Tool is called by LLM itself

In [49]:
while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    # Step 1: First LLM call
    response = agent.invoke(
        {"question": query},
        config={"configurable": {"session_id": "user1"}}
    )

    # Step 2: Check if tool is called
    if hasattr(response, "tool_calls") and response.tool_calls:

        tool_call = response.tool_calls[0]

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        # Step 3: Execute tool
        if tool_name == "calculator":
            tool_result = calculator.invoke(tool_args["expression"])

        # Step 4: Send tool result back to LLM
        final_prompt = f"""
User question: {query}

Tool result: {tool_result}

Give the final answer.
"""

        final_response = llm.invoke(final_prompt)

        print("\nAgent:", final_response.content)

    else:
        # No tool used
        print("\nAgent:", response.content)


You: what is AI

Agent: [{'type': 'text', 'text': 'Artificial intelligence (AI) is a broad field of computer science that aims to create intelligent machines that can perform tasks that typically require human intelligence. This includes abilities like learning, problem-solving, understanding language, and perceiving the environment. Machine learning and deep learning are subfields of AI.', 'extras': {'signature': 'CpQDAb4+9vt9jwrpx6bodZ9dH3P/hggY80PDWsrteeQX9na97MKKhh9ctRIiuanK2sIs2RrO9AF+INB5g0+LbSCrGYZw0CT64VvS0PUtQb2sre4UXt8uBlFjQLaAiqaMdHIBOg3wHrLBBunKKvNhId6hIHSHhOEPO6bHBsxhvMWjx+BoxivCaBygpz92Ic9I3SU0XfmdBDXjHnyu/j+JHTGsfACBb9CWMRNv7MspodIeUGyeSYgdufLLwvkmKWGvD7DLpn76Rg9lmqHoqbOHoIfy4ZJrDr16L3vr0tjmu5QkbfRC6P/kz7Jgw0eDw1EgicwxXvBZYq3SLY1bm8OtOtdFIEhPk8TCKJliewHEYmqdogLGUuy70ShAiAn+eJrqJgkd5niGfYDg6pi2sRLrVO7Arfw+REQR5stwIIcVgX4XGFzI1wIe+1qUdyfEQ7fInLIjhAzEd3tb+uaRQXTypHUuRyJ+j7KjIlllEqS/UczdSbAea2IchZHuPvaVEOvAXCCmtPpsrl/t9Yq8QeImT2xwkcjDXPQ='}}]

You: exit
